In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, accuracy_score, classification_report
)
from imblearn.over_sampling import ADASYN

warnings.filterwarnings("ignore")  # optional; remove if you want full sklearn warnings

# -------------------------
# CONFIG
CSV_PATH = 'final_merged_cleaned_data.csv'
POS_CLASS = 'Serious/Fatal'          # class to treat as "positive" for recall/precision/f1
POS_CLASS_THRESHOLD = 0.4            # probability threshold to predict POS_CLASS
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42
STD_DDof = 1                         # ddof=1 -> sample standard deviation (common choice)
# -------------------------

# 1. Load and Prepare
df = pd.read_csv(CSV_PATH)
# Combine Fatal + Serious into one label as in your original code
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = list(le.classes_)  # e.g. ['Other', 'Serious/Fatal'] or vice versa

if POS_CLASS not in class_names:
    raise ValueError(f"POS_CLASS '{POS_CLASS}' not found in encoded classes: {class_names}")
pos_label_idx = int(np.where(np.array(class_names) == POS_CLASS)[0][0])
other_label_idx = 1 - pos_label_idx  # binary problem: other label index

# 2. Features
X_raw = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 3. Model definition
mlp = MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                    activation='relu', solver='adam',
                    max_iter=500, early_stopping=True, random_state=RANDOM_STATE)

cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
ada = ADASYN(random_state=RANDOM_STATE)

# 4. Run CV and capture per-fold metrics
per_fold_records = []
aggregate_cm = np.zeros((2, 2), dtype=int)
all_y_true = []
all_y_pred = []

fold_counter = 0

print(f"Running RepeatedStratifiedKFold: n_splits={N_SPLITS}, n_repeats={N_REPEATS} -> {N_SPLITS * N_REPEATS} folds total\n")

for train_idx, test_idx in cv.split(X_scaled, y):
    fold_counter += 1
    repeat_idx = (fold_counter - 1) // N_SPLITS  # 0,1,2
    fold_in_repeat = ((fold_counter - 1) % N_SPLITS) + 1  # 1..5

    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Resample training set with ADASYN
    X_train_res, y_train_res = ada.fit_resample(X_train, y_train)

    # Fit model
    mlp.fit(X_train_res, y_train_res)

    # Predict with threshold on positive-class probability
    probs_pos = mlp.predict_proba(X_test)[:, pos_label_idx]
    preds_labels = np.where(probs_pos > POS_CLASS_THRESHOLD, pos_label_idx, other_label_idx)

    # Accumulate for concatenated metrics
    all_y_true.extend(y_test.tolist())
    all_y_pred.extend(preds_labels.tolist())
    aggregate_cm += confusion_matrix(y_test, preds_labels, labels=[0, 1])

    # Compute fold metrics (positive class = POS_CLASS)
    bal_acc = balanced_accuracy_score(y_test, preds_labels)
    recall = recall_score(y_test, preds_labels, pos_label=pos_label_idx, zero_division=0)
    precision = precision_score(y_test, preds_labels, pos_label=pos_label_idx, zero_division=0)
    f1 = f1_score(y_test, preds_labels, pos_label=pos_label_idx, zero_division=0)
    acc = accuracy_score(y_test, preds_labels)

    per_fold_records.append({
        'fold_global': fold_counter,
        'repeat': repeat_idx + 1,
        'fold_in_repeat': fold_in_repeat,
        'n_test': len(y_test),
        'balanced_accuracy': bal_acc,
        'recall_pos': recall,
        'precision_pos': precision,
        'f1_pos': f1,
        'accuracy': acc
    })

# Convert to DataFrame
per_fold_df = pd.DataFrame(per_fold_records).sort_values(['repeat', 'fold_in_repeat']).reset_index(drop=True)

# Print per-fold table
pd.set_option('display.precision', 4)
print("=== Per-fold metrics (each row = one fold) ===")
print(per_fold_df.to_string(index=False))

# Save
per_fold_df.to_csv('per_fold_metrics.csv', index=False)

# -------------------------
# Per-repeat (group by repeat): mean and std across the 5 folds in each repeat
repeat_groups = per_fold_df.groupby('repeat')
repeat_summary = []

for repeat_val, g in repeat_groups:
    means = g[['balanced_accuracy', 'recall_pos', 'precision_pos', 'f1_pos']].mean()
    stds = g[['balanced_accuracy', 'recall_pos', 'precision_pos', 'f1_pos']].std(ddof=STD_DDof)
    repeat_summary.append({
        'repeat': repeat_val,
        'n_folds': len(g),
        'balanced_accuracy_mean': means['balanced_accuracy'],
        'balanced_accuracy_std': stds['balanced_accuracy'],
        'recall_mean': means['recall_pos'],
        'recall_std': stds['recall_pos'],
        'precision_mean': means['precision_pos'],
        'precision_std': stds['precision_pos'],
        'f1_mean': means['f1_pos'],
        'f1_std': stds['f1_pos']
    })

repeat_summary_df = pd.DataFrame(repeat_summary).sort_values('repeat').reset_index(drop=True)
print("\n=== Per-repeat summary (mean ± std over each group of 5 folds) ===")
print(repeat_summary_df.to_string(index=False))
repeat_summary_df.to_csv('per_repeat_summary.csv', index=False)

# -------------------------
# Overall across 15 folds (treating the 15 folds as 15 independent observations):
# mean and std across the 15 fold-level metrics
metrics_cols = ['balanced_accuracy', 'recall_pos', 'precision_pos', 'f1_pos']
overall_15fold_mean = per_fold_df[metrics_cols].mean()
overall_15fold_std = per_fold_df[metrics_cols].std(ddof=STD_DDof)

overall_15fold_summary = pd.DataFrame([{
    'method': 'mean_of_15_folds',
    'n_items': len(per_fold_df),
    'balanced_accuracy_mean': overall_15fold_mean['balanced_accuracy'],
    'balanced_accuracy_std': overall_15fold_std['balanced_accuracy'],
    'recall_mean': overall_15fold_mean['recall_pos'],
    'recall_std': overall_15fold_std['recall_pos'],
    'precision_mean': overall_15fold_mean['precision_pos'],
    'precision_std': overall_15fold_std['precision_pos'],
    'f1_mean': overall_15fold_mean['f1_pos'],
    'f1_std': overall_15fold_std['f1_pos']
}])

# -------------------------
# Average of the 3 repeat-means (compute mean of the 3 repeat means, and std of those 3 values)
repeat_means_df = pd.DataFrame([{
    'balanced_accuracy': row['balanced_accuracy_mean'],
    'recall_pos': row['recall_mean'],
    'precision_pos': row['precision_mean'],
    'f1_pos': row['f1_mean']
} for _, row in repeat_summary_df.iterrows()])

avg_of_3_repeats_mean = repeat_means_df.mean()
avg_of_3_repeats_std = repeat_means_df.std(ddof=STD_DDof)

avg_of_repeats_summary = pd.DataFrame([{
    'method': 'average_of_3_repeat_means',
    'n_items': len(repeat_means_df),
    'balanced_accuracy_mean': avg_of_3_repeats_mean['balanced_accuracy'],
    'balanced_accuracy_std': avg_of_3_repeats_std['balanced_accuracy'],
    'recall_mean': avg_of_3_repeats_mean['recall_pos'],
    'recall_std': avg_of_3_repeats_std['recall_pos'],
    'precision_mean': avg_of_3_repeats_mean['precision_pos'],
    'precision_std': avg_of_3_repeats_std['precision_pos'],
    'f1_mean': avg_of_3_repeats_mean['f1_pos'],
    'f1_std': avg_of_3_repeats_std['f1_pos']
}])

# -------------------------
# Concatenated (pooled) metrics across all 15 folds - computed from pooled predictions
concat_bal_acc = balanced_accuracy_score(all_y_true, all_y_pred)
concat_recall = recall_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)
concat_precision = precision_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)
concat_f1 = f1_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)

concatenated_summary = pd.DataFrame([{
    'method': 'concatenated_15_folds',
    'n_items': len(all_y_true),
    'balanced_accuracy_mean': concat_bal_acc,
    'balanced_accuracy_std': np.nan,  # no std for pooled single estimate
    'recall_mean': concat_recall,
    'recall_std': np.nan,
    'precision_mean': concat_precision,
    'precision_std': np.nan,
    'f1_mean': concat_f1,
    'f1_std': np.nan
}])

# -------------------------
# Final combined overall summary table
final_summary_df = pd.concat([overall_15fold_summary, avg_of_repeats_summary, concatenated_summary], ignore_index=True)
pd.set_option('display.float_format', '{:.4f}'.format)

print("\n=== FINAL OVERALL PERFORMANCE SUMMARY ===")
print(final_summary_df.to_string(index=False))
final_summary_df.to_csv('overall_summary_table.csv', index=False)

# -------------------------
# Classification report (from concatenated predictions)
print("\n=== Classification report (concatenated 15 folds) ===")
print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=4))

# -------------------------
# Plot aggregated confusion matrix (all folds combined)
plt.figure(figsize=(6, 5))
sns.heatmap(aggregate_cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Aggregated Confusion Matrix (all 15 folds)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

# Save the raw concatenated predictions for later inspection (optional)
concat_df = pd.DataFrame({'y_true': all_y_true, 'y_pred': all_y_pred})
concat_df.to_csv('concatenated_predictions.csv', index=False)

print("\nSaved CSVs: per_fold_metrics.csv, per_repeat_summary.csv, overall_summary_table.csv, concatenated_predictions.csv")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, accuracy_score, classification_report
)
from imblearn.over_sampling import ADASYN

warnings.filterwarnings("ignore")  # optional; remove if you want full sklearn warnings

# -------------------------
# CONFIG
CSV_PATH = 'final_merged_cleaned_data.csv'
POS_CLASS = 'Serious/Fatal'          # class to treat as "positive" for recall/precision/f1
POS_CLASS_THRESHOLD = 0.4            # probability threshold to predict POS_CLASS
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42
STD_DDof = 1                         # ddof=1 -> sample standard deviation (common choice)
# -------------------------

# 1. Load and Prepare
df = pd.read_csv(CSV_PATH)
# Combine Fatal + Serious into one label as in your original code
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = list(le.classes_)  # e.g. ['Other', 'Serious/Fatal'] or vice versa

if POS_CLASS not in class_names:
    raise ValueError(f"POS_CLASS '{POS_CLASS}' not found in encoded classes: {class_names}")
pos_label_idx = int(np.where(np.array(class_names) == POS_CLASS)[0][0])
other_label_idx = 1 - pos_label_idx  # binary problem: other label index

# 2. Features
X_raw = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 3. Model definition
mlp = MLPClassifier(hidden_layer_sizes=(64, 64), #64,64 was also used
                    activation='relu', solver='adam',
                    max_iter=500, early_stopping=True, random_state=RANDOM_STATE)

cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
ada = ADASYN(random_state=RANDOM_STATE)

# 4. Run CV and capture per-fold metrics
per_fold_records = []
aggregate_cm = np.zeros((2, 2), dtype=int)
all_y_true = []
all_y_pred = []

fold_counter = 0

print(f"Running RepeatedStratifiedKFold: n_splits={N_SPLITS}, n_repeats={N_REPEATS} -> {N_SPLITS * N_REPEATS} folds total\n")

for train_idx, test_idx in cv.split(X_scaled, y):
    fold_counter += 1
    repeat_idx = (fold_counter - 1) // N_SPLITS  # 0,1,2
    fold_in_repeat = ((fold_counter - 1) % N_SPLITS) + 1  # 1..5

    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Resample training set with ADASYN
    X_train_res, y_train_res = ada.fit_resample(X_train, y_train)

    # Fit model
    mlp.fit(X_train_res, y_train_res)

    # Predict with threshold on positive-class probability
    probs_pos = mlp.predict_proba(X_test)[:, pos_label_idx]
    preds_labels = np.where(probs_pos > POS_CLASS_THRESHOLD, pos_label_idx, other_label_idx)

    # Accumulate for concatenated metrics
    all_y_true.extend(y_test.tolist())
    all_y_pred.extend(preds_labels.tolist())
    aggregate_cm += confusion_matrix(y_test, preds_labels, labels=[0, 1])

    # Compute fold metrics (positive class = POS_CLASS)
    bal_acc = balanced_accuracy_score(y_test, preds_labels)
    recall = recall_score(y_test, preds_labels, pos_label=pos_label_idx, zero_division=0)
    precision = precision_score(y_test, preds_labels, pos_label=pos_label_idx, zero_division=0)
    f1 = f1_score(y_test, preds_labels, pos_label=pos_label_idx, zero_division=0)
    acc = accuracy_score(y_test, preds_labels)

    per_fold_records.append({
        'fold_global': fold_counter,
        'repeat': repeat_idx + 1,
        'fold_in_repeat': fold_in_repeat,
        'n_test': len(y_test),
        'balanced_accuracy': bal_acc,
        'recall_pos': recall,
        'precision_pos': precision,
        'f1_pos': f1,
        'accuracy': acc
    })

# Convert to DataFrame
per_fold_df = pd.DataFrame(per_fold_records).sort_values(['repeat', 'fold_in_repeat']).reset_index(drop=True)

# Print per-fold table
pd.set_option('display.precision', 4)
print("=== Per-fold metrics (each row = one fold) ===")
print(per_fold_df.to_string(index=False))

# Save
per_fold_df.to_csv('per_fold_metrics.csv', index=False)

# -------------------------
# Per-repeat (group by repeat): mean and std across the 5 folds in each repeat
repeat_groups = per_fold_df.groupby('repeat')
repeat_summary = []

for repeat_val, g in repeat_groups:
    means = g[['balanced_accuracy', 'recall_pos', 'precision_pos', 'f1_pos']].mean()
    stds = g[['balanced_accuracy', 'recall_pos', 'precision_pos', 'f1_pos']].std(ddof=STD_DDof)
    repeat_summary.append({
        'repeat': repeat_val,
        'n_folds': len(g),
        'balanced_accuracy_mean': means['balanced_accuracy'],
        'balanced_accuracy_std': stds['balanced_accuracy'],
        'recall_mean': means['recall_pos'],
        'recall_std': stds['recall_pos'],
        'precision_mean': means['precision_pos'],
        'precision_std': stds['precision_pos'],
        'f1_mean': means['f1_pos'],
        'f1_std': stds['f1_pos']
    })

repeat_summary_df = pd.DataFrame(repeat_summary).sort_values('repeat').reset_index(drop=True)
print("\n=== Per-repeat summary (mean ± std over each group of 5 folds) ===")
print(repeat_summary_df.to_string(index=False))
repeat_summary_df.to_csv('per_repeat_summary.csv', index=False)

# -------------------------
# Overall across 15 folds (treating the 15 folds as 15 independent observations):
# mean and std across the 15 fold-level metrics
metrics_cols = ['balanced_accuracy', 'recall_pos', 'precision_pos', 'f1_pos']
overall_15fold_mean = per_fold_df[metrics_cols].mean()
overall_15fold_std = per_fold_df[metrics_cols].std(ddof=STD_DDof)

overall_15fold_summary = pd.DataFrame([{
    'method': 'mean_of_15_folds',
    'n_items': len(per_fold_df),
    'balanced_accuracy_mean': overall_15fold_mean['balanced_accuracy'],
    'balanced_accuracy_std': overall_15fold_std['balanced_accuracy'],
    'recall_mean': overall_15fold_mean['recall_pos'],
    'recall_std': overall_15fold_std['recall_pos'],
    'precision_mean': overall_15fold_mean['precision_pos'],
    'precision_std': overall_15fold_std['precision_pos'],
    'f1_mean': overall_15fold_mean['f1_pos'],
    'f1_std': overall_15fold_std['f1_pos']
}])

# -------------------------
# Average of the 3 repeat-means (compute mean of the 3 repeat means, and std of those 3 values)
repeat_means_df = pd.DataFrame([{
    'balanced_accuracy': row['balanced_accuracy_mean'],
    'recall_pos': row['recall_mean'],
    'precision_pos': row['precision_mean'],
    'f1_pos': row['f1_mean']
} for _, row in repeat_summary_df.iterrows()])

avg_of_3_repeats_mean = repeat_means_df.mean()
avg_of_3_repeats_std = repeat_means_df.std(ddof=STD_DDof)

avg_of_repeats_summary = pd.DataFrame([{
    'method': 'average_of_3_repeat_means',
    'n_items': len(repeat_means_df),
    'balanced_accuracy_mean': avg_of_3_repeats_mean['balanced_accuracy'],
    'balanced_accuracy_std': avg_of_3_repeats_std['balanced_accuracy'],
    'recall_mean': avg_of_3_repeats_mean['recall_pos'],
    'recall_std': avg_of_3_repeats_std['recall_pos'],
    'precision_mean': avg_of_3_repeats_mean['precision_pos'],
    'precision_std': avg_of_3_repeats_std['precision_pos'],
    'f1_mean': avg_of_3_repeats_mean['f1_pos'],
    'f1_std': avg_of_3_repeats_std['f1_pos']
}])

# -------------------------
# Concatenated (pooled) metrics across all 15 folds - computed from pooled predictions
concat_bal_acc = balanced_accuracy_score(all_y_true, all_y_pred)
concat_recall = recall_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)
concat_precision = precision_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)
concat_f1 = f1_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)

concatenated_summary = pd.DataFrame([{
    'method': 'concatenated_15_folds',
    'n_items': len(all_y_true),
    'balanced_accuracy_mean': concat_bal_acc,
    'balanced_accuracy_std': np.nan,  # no std for pooled single estimate
    'recall_mean': concat_recall,
    'recall_std': np.nan,
    'precision_mean': concat_precision,
    'precision_std': np.nan,
    'f1_mean': concat_f1,
    'f1_std': np.nan
}])

# -------------------------
# Final combined overall summary table
final_summary_df = pd.concat([overall_15fold_summary, avg_of_repeats_summary, concatenated_summary], ignore_index=True)
pd.set_option('display.float_format', '{:.4f}'.format)

print("\n=== FINAL OVERALL PERFORMANCE SUMMARY ===")
print(final_summary_df.to_string(index=False))
final_summary_df.to_csv('overall_summary_table.csv', index=False)

# -------------------------
# Classification report (from concatenated predictions)
print("\n=== Classification report (concatenated 15 folds) ===")
print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=4))

# -------------------------
# Plot aggregated confusion matrix (all folds combined)
plt.figure(figsize=(6, 5))
sns.heatmap(aggregate_cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Aggregated Confusion Matrix (all 15 folds)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

# Save the raw concatenated predictions for later inspection (optional)
concat_df = pd.DataFrame({'y_true': all_y_true, 'y_pred': all_y_pred})
concat_df.to_csv('concatenated_predictions.csv', index=False)

print("\nSaved CSVs: per_fold_metrics.csv, per_repeat_summary.csv, overall_summary_table.csv, concatenated_predictions.csv")


In [ ]:
# model_comparison_improve_model.py
# Try improved MLP + RandomForest + LightGBM (if available)
# Keeps the exact same metrics/tables as before (per-fold, per-repeat, overall, concatenated)
# Requires: scikit-learn, imbalanced-learn, pandas, numpy, matplotlib, seaborn, optionally lightgbm

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.metrics import (
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, accuracy_score, classification_report
)
from imblearn.over_sampling import ADASYN

# Models
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception:
    LGB_AVAILABLE = False

# -------------------------
# CONFIG (tweak as needed)
CSV_PATH = 'final_merged_cleaned_data.csv'
POS_CLASS = 'Serious/Fatal'
N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42
STD_DDOF = 1           # sample std
K_BEST = 10            # feature selection
ADASYNC_RANDOM_STATE = 42

# Models to try (you can add/remove)
MODELS = {}

# Improved MLP: deeper + stronger regularization + smaller LR
MODELS['Improved_MLP'] = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=5e-4,
    alpha=1e-3,            # stronger regularization
    max_iter=300,          # key speedup
    early_stopping=True,
    n_iter_no_change=15,
    random_state=RANDOM_STATE
)


# Random Forest: strong tree baseline (class-balanced)
MODELS['RandomForest'] = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    class_weight='balanced_subsample'
)

# LightGBM (if available)
if LGB_AVAILABLE:
    MODELS['LightGBM'] = lgb.LGBMClassifier(
        objective='binary',
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

# -------------------------
# 1. Load & prepare
df = pd.read_csv(CSV_PATH)
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = list(le.classes_)

if POS_CLASS not in class_names:
    raise ValueError(f"POS_CLASS '{POS_CLASS}' not found in classes: {class_names}")
pos_label_idx = class_names.index(POS_CLASS)

# 2. Feature selection (same as you used)
X_raw = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())

print(f"Selecting top {K_BEST} features by mutual information...")
selector = SelectKBest(score_func=mutual_info_classif, k=K_BEST)
X_selected = selector.fit_transform(X_raw, y)
selected_features = X_raw.columns[selector.get_support()]
print(f"Selected features: {list(selected_features)}")

# 3. Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

# 4. CV setup & ADASYN
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
ada = ADASYN(random_state=ADASYNC_RANDOM_STATE)

# Helper: run CV for a single model and produce the same metric tables & CSVs
def run_model_cv(model_name, model_obj, X, y, pos_label_idx, cv, ada, std_ddof=1):
    print("\n" + "="*80)
    print(f"Running model: {model_name}")
    per_fold_records = []
    aggregate_cm = np.zeros((2,2), dtype=int)
    all_y_true, all_y_pred = [], []
    accuracies = []
    fold_counter = 0
    total_folds = (cv.get_n_splits())
    # sklearn's RepeatedStratifiedKFold get_n_splits returns n_splits; but we know total = n_splits*n_repeats
    # We'll just count folds as we iterate

    for train_idx, test_idx in cv.split(X, y):
        fold_counter += 1
        repeat_idx = (fold_counter - 1) // N_SPLITS
        fold_in_repeat = ((fold_counter - 1) % N_SPLITS) + 1


        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # ADASYN on training data (consistent across models)
        X_train_res, y_train_res = ada.fit_resample(X_train, y_train)

        # For LightGBM, we can pass numpy arrays directly. For some models, warm-start is off by default.
        model = model_obj  # instantiate if needed (we assume model_obj is a fresh estimator)
        # Fit
        if model_name == 'LightGBM' and LGB_AVAILABLE:
            # use eval_set for early stopping inside LightGBM
            model.fit(X_train_res, y_train_res,
                      eval_set=[(X_test, y_test)],
                      eval_metric='binary_logloss',
                      early_stopping_rounds=50,
                      verbose=False)
        else:
            model.fit(X_train_res, y_train_res)

        # Prediction (default decision)
        preds = model.predict(X_test)
        # ensure shape and type
        preds = np.asarray(preds).astype(int).ravel()

        # Accumulate
        all_y_true.extend(y_test.tolist())
        all_y_pred.extend(preds.tolist())
        aggregate_cm += confusion_matrix(y_test, preds, labels=[0,1])
        accuracies.append(accuracy_score(y_test, preds))

        # Per-fold metrics
        bal_acc = balanced_accuracy_score(y_test, preds)
        recall = recall_score(y_test, preds, pos_label=pos_label_idx, zero_division=0)
        precision = precision_score(y_test, preds, pos_label=pos_label_idx, zero_division=0)
        f1 = f1_score(y_test, preds, pos_label=pos_label_idx, zero_division=0)
        acc = accuracy_score(y_test, preds)

        per_fold_records.append({
            'fold_global': fold_counter,
            'repeat': repeat_idx + 1,
            'fold_in_repeat': fold_in_repeat,
            'n_test': len(y_test),
            'balanced_accuracy': bal_acc,
            'recall_pos': recall,
            'precision_pos': precision,
            'f1_pos': f1,
            'accuracy': acc
        })

        print(f"  {model_name} - Completed fold {fold_counter} (repeat {repeat_idx+1}, fold {fold_in_repeat})")

    # Convert to DataFrame
    per_fold_df = pd.DataFrame(per_fold_records).sort_values(['repeat','fold_in_repeat']).reset_index(drop=True)
    per_fold_df.to_csv(f'per_fold_metrics_{model_name}.csv', index=False)

    # Per-repeat summary
    repeat_summary = []
    for repeat_val, g in per_fold_df.groupby('repeat'):
        means = g[['balanced_accuracy','recall_pos','precision_pos','f1_pos']].mean()
        stds = g[['balanced_accuracy','recall_pos','precision_pos','f1_pos']].std(ddof=std_ddof)
        repeat_summary.append({
            'repeat': repeat_val,
            'n_folds': len(g),
            'balanced_accuracy_mean': means['balanced_accuracy'],
            'balanced_accuracy_std': stds['balanced_accuracy'],
            'recall_mean': means['recall_pos'],
            'recall_std': stds['recall_pos'],
            'precision_mean': means['precision_pos'],
            'precision_std': stds['precision_pos'],
            'f1_mean': means['f1_pos'],
            'f1_std': stds['f1_pos']
        })
    repeat_summary_df = pd.DataFrame(repeat_summary).sort_values('repeat').reset_index(drop=True)
    repeat_summary_df.to_csv(f'per_repeat_summary_{model_name}.csv', index=False)

    # Overall 15-fold mean & std (across the 15 fold-level metrics)
    metrics_cols = ['balanced_accuracy','recall_pos','precision_pos','f1_pos']
    overall_15_mean = per_fold_df[metrics_cols].mean()
    overall_15_std  = per_fold_df[metrics_cols].std(ddof=std_ddof)
    overall_15_summary = pd.DataFrame([{
        'method':'mean_of_15_folds',
        'n_items': len(per_fold_df),
        'balanced_accuracy_mean': overall_15_mean['balanced_accuracy'],
        'balanced_accuracy_std': overall_15_std['balanced_accuracy'],
        'recall_mean': overall_15_mean['recall_pos'],
        'recall_std': overall_15_std['recall_pos'],
        'precision_mean': overall_15_mean['precision_pos'],
        'precision_std': overall_15_std['precision_pos'],
        'f1_mean': overall_15_mean['f1_pos'],
        'f1_std': overall_15_std['f1_pos']
    }])

    # Average of the 3 repeat-means
    repeat_means_df = pd.DataFrame([{
        'balanced_accuracy': r['balanced_accuracy_mean'],
        'recall_pos': r['recall_mean'],
        'precision_pos': r['precision_mean'],
        'f1_pos': r['f1_mean']
    } for _, r in repeat_summary_df.iterrows()])
    avg_repeats_mean = repeat_means_df.mean()
    avg_repeats_std  = repeat_means_df.std(ddof=std_ddof)
    avg_repeats_summary = pd.DataFrame([{
        'method':'average_of_3_repeat_means',
        'n_items': len(repeat_means_df),
        'balanced_accuracy_mean': avg_repeats_mean['balanced_accuracy'],
        'balanced_accuracy_std': avg_repeats_std['balanced_accuracy'],
        'recall_mean': avg_repeats_mean['recall_pos'],
        'recall_std': avg_repeats_std['recall_pos'],
        'precision_mean': avg_repeats_mean['precision_pos'],
        'precision_std': avg_repeats_std['precision_pos'],
        'f1_mean': avg_repeats_mean['f1_pos'],
        'f1_std': avg_repeats_std['f1_pos']
    }])

    # Concatenated pooled metrics (point estimates only)
    concat_bal_acc = balanced_accuracy_score(all_y_true, all_y_pred)
    concat_recall  = recall_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)
    concat_precision = precision_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)
    concat_f1 = f1_score(all_y_true, all_y_pred, pos_label=pos_label_idx, zero_division=0)
    concatenated_summary = pd.DataFrame([{
        'method':'concatenated_15_folds',
        'n_items': len(all_y_true),
        'balanced_accuracy_mean': concat_bal_acc,
        'balanced_accuracy_std': np.nan,
        'recall_mean': concat_recall,
        'recall_std': np.nan,
        'precision_mean': concat_precision,
        'precision_std': np.nan,
        'f1_mean': concat_f1,
        'f1_std': np.nan
    }])

    final_summary_df = pd.concat([overall_15_summary, avg_repeats_summary, concatenated_summary], ignore_index=True)
    final_summary_df.to_csv(f'overall_summary_{model_name}.csv', index=False)

    # Save concatenated predictions
    concat_df = pd.DataFrame({'y_true': all_y_true, 'y_pred': all_y_pred})
    concat_df.to_csv(f'concatenated_predictions_{model_name}.csv', index=False)

    # Print outputs
    pd.set_option('display.precision',4)
    print("\n--- Per-fold (first 15 rows) ---")
    print(per_fold_df.to_string(index=False))
    print("\n--- Per-repeat summary ---")
    print(repeat_summary_df.to_string(index=False))
    print("\n--- Final overall summary ---")
    print(final_summary_df.to_string(index=False))
    print("\n--- Classification report (concatenated predictions) ---")
    print(classification_report(all_y_true, all_y_pred, target_names=class_names, digits=4))

    # Plot aggregated confusion matrix
    plt.figure(figsize=(6,5))
    sns.heatmap(aggregate_cm, annot=True, fmt='d', cmap='viridis', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Aggregated Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

    return {
        'model': model_name,
        'per_fold_df': per_fold_df,
        'repeat_summary_df': repeat_summary_df,
        'final_summary_df': final_summary_df,
        'concatenated_preds': concat_df,
        'aggregate_cm': aggregate_cm,
        'overall_mean_balanced_accuracy': overall_15_mean['balanced_accuracy'],
        'concatenated_balanced_accuracy': concat_bal_acc
    }

# -------------------------
# Run models
results = []
for model_name, model_obj in MODELS.items():
    if model_name == 'LightGBM' and not LGB_AVAILABLE:
        print("\nLightGBM not available; skipping LightGBM. Install 'lightgbm' to enable it.")
        continue
    # clone model to avoid reusing fitted estimator across folds
    from sklearn.base import clone
    model_clone = clone(model_obj)
    res = run_model_cv(model_name, model_clone, X_scaled, y, pos_label_idx, cv, ada, STD_DDOF)
    results.append(res)

# Summary comparison (balanced accuracy mean_of_15_folds)
summary_comp = pd.DataFrame([{
    'model': r['model'],
    'mean_of_15_folds_balanced_accuracy': r['overall_mean_balanced_accuracy'],
    'concatenated_balanced_accuracy': r['concatenated_balanced_accuracy']
} for r in results])

print("\n" + "="*80)
print("Model comparison (balanced accuracy):")
print(summary_comp.to_string(index=False))
summary_comp.to_csv('model_comparison_summary.csv', index=False)

print("\nDone. CSVs saved for each model: per_fold_metrics_<model>.csv, per_repeat_summary_<model>.csv, overall_summary_<model>.csv, concatenated_predictions_<model>.csv")
